# Trace export with source line identities

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from pathlib import Path
import csv,json,pickle,hashlib,sys,math
import numpy as np
from contract import ROOT as REPO,REFERENCE,CODE,PROV,read_json,write_json,sha,verify_inputs
PHASE=REFERENCE
ROOT=REPO/'outputs/analysis'
ROOT.mkdir(parents=True,exist_ok=True)
def csv_out(path,rows):
    with path.open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def csvread(path):
    with Path(path).open() as f:return list(csv.DictReader(f))
read=read_json
write=write_json

def export_traces():
    OUT=REPO/'outputs/exports/solid_rally_development';OUT.mkdir(parents=True,exist_ok=True)
    sources=[];traces=[];groups={};pair_count=0
    with (OUT/'valid_window_trajectories.jsonl').open('w') as export:
        for condition in ['TASK','MAX','RANDOM']:
            attempt=PHASE/'runs'/('P1-'+condition)/'attempt-01';cfg=read(attempt/'resolved_config.json')
            stages=(['train'] if condition!='RANDOM' else [])+[f'evaluate-{s}' for s in range(3001,3011)]
            for stage in stages:
                d=attempt/stage;result=read(d/'result.json');assert result['status']=='PASS'
                path=d/'events.jsonl';source_hash=sha(path);cfg_hash=sha(attempt/'resolved_config.json')
                sources.append({'path':str(path.relative_to(PHASE)),'bytes':path.stat().st_size,'sha256':source_hash,'config_path':str((attempt/'resolved_config.json').relative_to(PHASE)),'config_sha256':cfg_hash})
                split='training' if stage=='train' else 'evaluation';bucket=groups.setdefault((condition,split),{'q':[],'clipped':0,'traces':set(),'pair_hashes':set()})
                per_episode={}
                with path.open() as f:
                    for line_number,line in enumerate(f,1):
                        r=json.loads(line)
                        if r['event']!='decision' or not r['fresh']:continue
                        assert r['valid'] and r['decision']>=30 and r['decision']%15==0
                        assert r['pair_id']==[r['episode'],r['decision']//15-1,r['decision']//15]
                        assert len(r['pair_input'])==54 and all(math.isfinite(x) for x in r['pair_input']) and 0<=r['q']<=1
                        key=f"P1-{condition}/attempt-01/{stage}/episode-{r['episode']}"
                        e=per_episode.setdefault(r['episode'],{'trace_id':key,'q':[],'clipped':0,'hash':hashlib.sha256(),'ids':set()})
                        assert tuple(r['pair_id']) not in e['ids'];e['ids'].add(tuple(r['pair_id']))
                        e['hash'].update(json.dumps([r['decision'],r['q'],r['pair_input']],separators=(',',':')).encode())
                        clipped=r['estimator']['clipped_member_features'];e['q'].append(r['q']);e['clipped']+=int(clipped>0)
                        bucket['q'].append(r['q']);bucket['clipped']+=int(clipped>0);bucket['traces'].add(key)
                        record={'trace_id':key,'condition':condition,'split':split,'episode':r['episode'],'decision':r['decision'],
                            'pair_id':r['pair_id'],'q_increase':r['q'],'pair_input_L_then_R':r['pair_input'],
                            'window_intervals_seconds':r['pair_window_intervals_seconds'],'comparison_game_seconds':r['comparison_game_seconds'],
                            'availability_game_seconds':r['availability_game_seconds'],'raw_score':r['raw_score_signal'],'task_event':r['b'],
                            'delivered_reward':r['delivered_reward'],'clipped_member_features':clipped,'valid':True,'fresh':True,
                            'source_events':str(path.relative_to(PHASE)),'source_line':line_number,'source_sha256':source_hash,'config_sha256':cfg_hash}
                        export.write(json.dumps(record,separators=(',',':'),allow_nan=False)+'\n');pair_count+=1
                for summary in read(d/'episode_summaries.json'):
                    e=per_episode[summary['episode']];assert len(e['q'])==summary['pairs']==max(summary['decisions']//15-1,0)
                    digest=e['hash'].hexdigest();bucket['pair_hashes'].add(digest)
                    traces.append({'trace_id':e['trace_id'],'condition':condition,'split':split,'stage':stage,'episode':summary['episode'],
                        'requested_Unity_seed':result['simulator_seed'],'action_sampler_seed':result.get('action_sampler_seed'),
                        'policy_initialization_seed':cfg['policy_initialization_seed'],'decisions':summary['decisions'],'valid_pairs':len(e['q']),
                        'q_min':min(e['q']),'q_median':float(np.median(e['q'])),'q_max':max(e['q']),'q_mean':float(np.mean(e['q'])),
                        'pairs_with_any_clipping':e['clipped'],'task_events':summary['task_return'],'raw_final_score':summary['raw_score'],
                        'terminated':summary['terminated'],'truncated':summary['truncated'],'end_reason':summary['end_reason'],
                        'pair_sequence_sha256':digest,'config_sha256':cfg_hash,'source_sha256':source_hash})
    support=[]
    for (c,split),g in groups.items():
        qs=np.asarray(g['q']);support.append({'condition':c,'split':split,'traces_including_fragments':len(g['traces']),
            'distinct_pair_sequences':len(g['pair_hashes']),'pairs':len(qs),'q_min':float(qs.min()),'q_p05':float(np.quantile(qs,.05)),
            'q_p25':float(np.quantile(qs,.25)),'q_median':float(np.median(qs)),'q_p75':float(np.quantile(qs,.75)),
            'q_p95':float(np.quantile(qs,.95)),'q_max':float(qs.max()),'q_mean':float(qs.mean()),
            'pairs_with_any_clipping':g['clipped'],'clipping_pair_fraction':g['clipped']/len(qs)})
    assert pair_count==7824 and len(traces)==202
    csv_out(OUT/'trace_index.csv',traces);csv_out(OUT/'support_summary.csv',support);write(OUT/'source_manifest.json',sources)

    # Re-read serialized identities independently of the in-memory counters.
    counts={};seen=set()
    with (OUT/'valid_window_trajectories.jsonl').open() as stream:
        for line in stream:
            record=json.loads(line);identity=(record['trace_id'],tuple(record['pair_id']))
            if identity in seen:raise ValueError('Duplicate serialized pair')
            seen.add(identity);counts[record['trace_id']]=counts.get(record['trace_id'],0)+1
    assert len(seen)==7824 and len(counts)==202
    assert all(counts[t['trace_id']]==t['valid_pairs'] for t in traces)
    manifest=[{'path':str(p.relative_to(REPO)),'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(OUT.iterdir()) if p.is_file() and p.name!='artifact_manifest.json']
    write(OUT/'artifact_manifest.json',manifest)
    return {'pairs':len(seen),'traces':len(counts),'source_event_files':len(sources),'output':str(OUT)}
print('Trace export with source line identities definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Trace export with source line identities definitions/execution completed.
